# Speech Graph Metrics — Correlation Analysis

## Overview

This notebook analyzes correlations between speech graph metrics and verbal production variables. Metrics are loaded from `data_SpeechGraph2.csv`, the precomputed output of `visualization.ipynb`.

| Metric | Definition |
|---|---|
| **n_tokens** | Total number of tokens (gross speech length) |
| **n_nodes** | Number of unique words (lexical diversity proxy) |
| **n_edges** | Number of unique word transitions |
| **LSCC** | Largest Strongly Connected Component size |
| **AD** | Average Degree (edges / nodes) |
| **ASPL** | Average Shortest Path Length within the LSCC |

### Research Questions
1. How strongly are LSCC, AD, and ASPL intercorrelated?
2. Do graph metrics correlate with production variables (n_tokens, n_nodes, n_edges)?
3. Do these correlations differ between SSD and Control groups?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import pearsonr

mpl.rcParams['font.family'] = 'Times New Roman'
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')

## 1. Load Precomputed Metrics

Metrics are loaded from `data_SpeechGraph2.csv`, the output of `visualization.ipynb`. That notebook iterates over all verbatims, builds a directed speech graph for each, and computes the six retained metrics.

In [ ]:
OUTPUT_FILE = '../SpeechGraph/data_SpeechGraph2.csv'

df = pd.read_csv(OUTPUT_FILE, sep=';')

# Rename metric columns to internal naming convention
col_rename = {'LSCC': 'sg_lscc_size', 'AD': 'sg_avg_degree', 'ASPL': 'sg_aspl'}
df = df.rename(columns=col_rename)

# Detect participant ID column
id_col = 'No anonyme' if 'No anonyme' in df.columns else 'No'

# Filter for SSD (100) and Control (300)
if 'Groupe' in df.columns:
    df = df[df['Groupe'].isin([100, 300])].copy()
    df['Groupe'] = df['Groupe'].astype(int)

# Retained metrics
graph_cols = ['sg_lscc_size', 'sg_avg_degree', 'sg_aspl']
metric_cols = ['n_tokens', 'n_nodes', 'n_edges'] + graph_cols

print(f'Data loaded: {len(df)} verbatims')
print(f'Columns: {list(df.columns)}')
if 'Groupe' in df.columns:
    print(f'Groups: SSD={sum(df.Groupe==100)}, Control={sum(df.Groupe==300)}')
print(f'\nRetained metrics: {metric_cols}')
print(df[metric_cols].describe().round(3))

## 2. Descriptive Statistics

Metrics aggregated per participant (mean across stories).

In [ ]:
labels = {100: 'SSD', 300: 'Control'}

group_cols = [c for c in [id_col, 'Groupe'] if c in df.columns]
df_part = (
    df.groupby(group_cols)[[c for c in metric_cols if c in df.columns]]
    .mean()
    .reset_index()
)

print('=== Descriptive Statistics by Group ===\n')
for metric in metric_cols:
    if metric not in df_part.columns:
        continue
    print(f'{metric}:')
    if 'Groupe' in df_part.columns:
        for grp in [100, 300]:
            vals = df_part[df_part['Groupe'] == grp][metric].dropna()
            print(f'  {labels[grp]:10s} : M={vals.mean():.2f}  SD={vals.std():.2f}  Md={vals.median():.2f}  n={len(vals)}')
    else:
        vals = df_part[metric].dropna()
        print(f'  Overall : M={vals.mean():.2f}  SD={vals.std():.2f}  Md={vals.median():.2f}  n={len(vals)}')
    print()

## 3. Overall Correlation Matrix

In [ ]:
corr_cols = [c for c in metric_cols if c in df_part.columns]
df_corr = df_part[corr_cols].dropna()

corr_matrix = df_corr.corr(method='pearson')

pval_mat = pd.DataFrame(
    np.ones((len(corr_cols), len(corr_cols))),
    index=corr_cols, columns=corr_cols
)
for i, col1 in enumerate(corr_cols):
    for j, col2 in enumerate(corr_cols):
        if i <= j:
            r, p = pearsonr(df_corr[col1].dropna(), df_corr[col2].dropna())
            pval_mat.iloc[i, j] = p
            pval_mat.iloc[j, i] = p

print('=== Correlation Matrix (Pearson) ===\n')
print(corr_matrix.round(3))

label_map_full = {
    'n_tokens': 'Tokens', 'n_nodes': 'Nodes', 'n_edges': 'Edges',
    'sg_lscc_size': 'LSCC', 'sg_avg_degree': 'AD', 'sg_aspl': 'ASPL',
}
tick_labels_full = [label_map_full.get(c, c) for c in corr_cols]

fig, ax = plt.subplots(figsize=(8, 7))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=0)
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, vmin=-1, vmax=1, square=True, ax=ax,
            xticklabels=tick_labels_full, yticklabels=tick_labels_full,
            cbar_kws={'label': 'Pearson r'})
ax.set_title('Correlation Matrix: Graph & Production Metrics', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3b. Core Metrics: n_tokens, LSCC, AD, ASPL (Lower Triangular)

In [ ]:

mpl.rcParams['font.family'] = 'Times New Roman'
core_metrics = ['n_tokens', 'n_words'] + graph_cols
available_core = [m for m in core_metrics if m in df_part.columns]

print(f'Core metrics: {available_core}')

if len(available_core) >= 2:
    df_core = df_part[available_core].dropna()

    corr_core = df_core.corr(method='pearson')

    pval_core = pd.DataFrame(np.ones((len(available_core), len(available_core))),
                             index=available_core, columns=available_core)
    for i, col1 in enumerate(available_core):
        for j, col2 in enumerate(available_core):
            if i != j:
                r, p = pearsonr(df_core[col1].dropna(), df_core[col2].dropna())
                pval_core.iloc[i, j] = p

    print('\n=== 4x4 Correlation Matrix (Lower Triangular) ===\n')
    print(f"{'':20}", end='')
    for col in available_core:
        print(f'{col:>20}', end='')
    print()
    print('-' * (20 + 20 * len(available_core)))
    for i, row_idx in enumerate(available_core):
        print(f'{row_idx:20}', end='')
        for j, col_idx in enumerate(available_core):
            if i > j:
                r = corr_core.iloc[i, j]
                p = pval_core.iloc[i, j]
                sig = '**' if p < 0.01 else ('*' if p < 0.05 else ' ')
                print(f'{r:>18.3f}{sig} ', end='')
            else:
                print(f'{"":>20}', end='')
        print()

    # Clean axis labels
    label_map = {
        'n_tokens': 'n_words',
        'n_words': 'Words',
        'sg_lscc_size': 'LSCC',
        'sg_avg_degree': 'AD',
        'sg_aspl': 'ASPL',
    }
    tick_labels = [label_map.get(m, m) for m in available_core]

    # Build custom annotation matrix: "r = X.XX\np < .001" in each lower-triangle cell
    n = len(available_core)
    annot_matrix = np.full((n, n), '', dtype=object)
    for i in range(n):
        for j in range(n):
            if i > j:
                r_val = corr_core.iloc[i, j]
                p_val = pval_core.iloc[i, j]
                r_str = f'r = {r_val:.2f}'
                if p_val < 0.001:
                    p_str = 'p < .001'
                else:
                    p_str = f'p = {p_val:.3f}'.replace('0.', '.')
                annot_matrix[i, j] = f'{r_str}\n{p_str}'

    fig, ax = plt.subplots(figsize=(8, 7))
    mask = np.triu(np.ones_like(corr_core, dtype=bool), k=0)
    sns.heatmap(corr_core, mask=mask, annot=annot_matrix, fmt='', cmap='coolwarm',
                center=0, vmin=-1, vmax=1, square=True, ax=ax,
                cbar_kws={'label': 'Pearson r', 'shrink': 0.85},
                xticklabels=tick_labels, yticklabels=tick_labels,
                linewidths=2, linecolor='white',
                annot_kws={'size': 20})
    ax.tick_params(labelsize=25)
    ax.figure.axes[-1].tick_params(labelsize=25)
    ax.figure.axes[-1].set_ylabel('Pearson r', fontsize=25)
    plt.tight_layout()
    plt.show()

    print('\n=== Detailed Statistics ===')
    for i, metric in enumerate(available_core):
        print(f'\n{metric}:')
        for j in range(i):
            col_metric = available_core[j]
            r = corr_core.iloc[i, j]
            p = pval_core.iloc[i, j]
            n = len(df_core[[metric, col_metric]].dropna())
            sig = '**' if p < 0.01 else ('*' if p < 0.05 else 'ns')
            print(f'  vs {col_metric:25s}: r = {r:7.3f}, p = {p:.4f} ({sig}), n = {n}')
else:
    print(f'Need at least 2 metrics. Available: {available_core}')


## 4. Group-Specific Correlations

In [ ]:
def fisher_z_test(r1, n1, r2, n2):
    """Fisher's z-transformation to test if two correlations differ significantly."""
    z1 = np.arctanh(r1)
    z2 = np.arctanh(r2)
    se = np.sqrt(1 / (n1 - 3) + 1 / (n2 - 3))
    z_score = (z1 - z2) / se if se > 0 else 0
    p_value = 2 * (1 - stats.norm.cdf(abs(z_score)))
    return z_score, p_value

corr_by_group = {}
if 'Groupe' in df_part.columns:
    for grp in [100, 300]:
        df_grp = df_part[df_part['Groupe'] == grp][corr_cols].dropna()
        corr_by_group[grp] = df_grp.corr(method='pearson')

    print('SSD (Group 100):')
    print(corr_by_group[100].round(3))
    print('\nControl (Group 300):')
    print(corr_by_group[300].round(3))

    lbl = {
        'sg_lscc_size': 'LSCC', 'sg_avg_degree': 'AD', 'sg_aspl': 'ASPL',
        'n_tokens': 'n_tokens', 'n_nodes': 'n_nodes', 'n_edges': 'n_edges',
    }
    key_pairs = [
        ('sg_lscc_size', 'n_tokens'), ('sg_avg_degree', 'n_tokens'), ('sg_aspl', 'n_tokens'),
        ('sg_lscc_size', 'n_nodes'),  ('sg_lscc_size', 'n_edges'),
    ]

    print('\n=== Fisher z-Test: Correlation Differences Between Groups ===\n')
    print(f"{'Pair':<35} {'r_SSD':>8} {'r_CO':>8} {'z':>8} {'p':>8}")
    print('-' * 65)
    for var1, var2 in key_pairs:
        if var1 not in corr_cols or var2 not in corr_cols:
            continue
        r_ssd = corr_by_group[100].loc[var1, var2]
        n_ssd = len(df_part[df_part['Groupe'] == 100][var1].dropna())
        r_co  = corr_by_group[300].loc[var1, var2]
        n_co  = len(df_part[df_part['Groupe'] == 300][var1].dropna())
        if pd.isna(r_ssd) or pd.isna(r_co):
            continue
        z_score, p_value = fisher_z_test(r_ssd, n_ssd, r_co, n_co)
        sig = '*' if p_value < 0.05 else ''
        pair_label = f'{lbl.get(var1, var1)} — {lbl.get(var2, var2)}'
        print(f'{pair_label:<35} {r_ssd:>8.3f} {r_co:>8.3f} {z_score:>8.2f} {p_value:>7.3f}{sig}')
else:
    print('No Groupe column found — group-specific analysis skipped.')

## 5. Summary

In [ ]:
lbl = {
    'sg_lscc_size': 'LSCC', 'sg_avg_degree': 'AD', 'sg_aspl': 'ASPL',
    'n_tokens': 'n_tokens', 'n_nodes': 'n_nodes', 'n_edges': 'n_edges',
}

print('=' * 70)
print('GRAPH METRICS & PRODUCTION VARIABLES')
print('=' * 70)

print('\n1. INTERCORRELATIONS AMONG GRAPH METRICS')
print('-' * 70)
for i, col1 in enumerate(graph_cols):
    for col2 in graph_cols[i + 1:]:
        if col1 not in corr_matrix.columns or col2 not in corr_matrix.columns:
            continue
        r = corr_matrix.loc[col1, col2]
        p = pval_mat.loc[col1, col2]
        sig = '**' if p < 0.01 else ('*' if p < 0.05 else '')
        print(f'{lbl.get(col1):6s} ↔ {lbl.get(col2):6s} : r = {r:7.3f} (p={p:.4f}){sig}')

print('\n2. GRAPH METRICS ↔ PRODUCTION VARIABLES')
print('-' * 70)
for metric in graph_cols:
    if metric not in corr_matrix.columns:
        continue
    print(f'\n{lbl.get(metric, metric)}:')
    for var in ['n_tokens', 'n_nodes', 'n_edges']:
        if var not in corr_matrix.columns:
            continue
        r = corr_matrix.loc[metric, var]
        p = pval_mat.loc[metric, var]
        sig = '**' if p < 0.01 else ('*' if p < 0.05 else '')
        print(f'  vs {var:10s} : r = {r:7.3f} (p={p:.4f}){sig}')

if corr_by_group:
    print('\n3. GROUP DIFFERENCES (Fisher z-test)')
    print('-' * 70)
    for var1, var2 in [
        ('sg_lscc_size', 'n_tokens'),
        ('sg_avg_degree', 'n_tokens'),
        ('sg_aspl', 'n_tokens'),
    ]:
        if var1 not in corr_cols or var2 not in corr_cols:
            continue
        r_ssd = corr_by_group[100].loc[var1, var2]
        n_ssd = len(df_part[df_part['Groupe'] == 100][var1].dropna())
        r_co  = corr_by_group[300].loc[var1, var2]
        n_co  = len(df_part[df_part['Groupe'] == 300][var1].dropna())
        if pd.isna(r_ssd) or pd.isna(r_co):
            continue
        z_score, p_value = fisher_z_test(r_ssd, n_ssd, r_co, n_co)
        sig = ' *' if p_value < 0.05 else ''
        print(f'{lbl.get(var1)} vs {var2}: z={z_score:6.2f}, p={p_value:.4f}{sig}')
        print(f'  SSD: r={r_ssd:6.3f} (n={n_ssd}), Control: r={r_co:6.3f} (n={n_co})')

print('\n' + '=' * 70)